# Clase 26 · Notebook A: Etiquetado con Label Studio

**Diplomado en Data Science Aplicada con Python** · Arca Continental Ecuador x UDLA

---

Este notebook es **solo para la fase de etiquetado**. Está separado del notebook de training porque Label Studio actualiza NumPy y rompe la instalación de TensorFlow.

**Flujo:**

1. **Notebook A** (este): el profesor levanta Label Studio + localtunnel, los estudiantes etiquetan en equipo. Exportamos el dataset.
2. **Notebook B** (`Clase_26_Detection.ipynb`): cargamos el dataset etiquetado, fine-tuneamos YOLO26, agregamos OCR y desplegamos Gradio.

> **Importante**: este notebook NO necesita GPU (es solo un servidor web). Puedes correrlo en CPU para reservar GPU para el otro notebook.

## 1. Instalar dependencias

> ⚠️ Label Studio actualiza numpy/pandas a versiones nuevas. **No mezcles este notebook con código TF/Keras** — usa el Notebook B para eso.

In [ ]:
!pip install -q label-studio

In [ ]:
!npm install -g localtunnel 2>&1 | tail -3
print("localtunnel instalado")

## 2. Levantar Label Studio en background

In [ ]:
import os, threading, time, subprocess

# Configurar Label Studio
os.environ["LABEL_STUDIO_LOCAL_FILES_SERVING_ENABLED"] = "true"
os.environ["LABEL_STUDIO_DISABLE_SIGNUP_WITHOUT_LINK"] = "false"
os.environ["LABEL_STUDIO_BASE_DATA_DIR"] = "/content/label-studio-data"

# Lanzar en background
def run_ls():
    os.system("label-studio start --port 8080 --host 0.0.0.0 --no-browser")

threading.Thread(target=run_ls, daemon=True).start()
print("Iniciando Label Studio (~30s)...")
time.sleep(30)
print("OK. Listo para exponer con localtunnel.")

## 3. Exponer con localtunnel (sin token)

`localtunnel` (`lt`) es la alternativa open-source a ngrok. **No requiere registrarse**: corre y devuelve la URL pública.

In [ ]:
# Lanzar localtunnel y capturar la URL pública del stdout
import subprocess, re, threading

estado = {"url": None, "proc": None}

def lanzar_localtunnel():
    proc = subprocess.Popen(
        ["lt", "--port", "8080"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    estado["proc"] = proc
    for linea in proc.stdout:
        print(linea, end="")
        m = re.search(r"(https://[a-z0-9-]+\.loca\.lt)", linea)
        if m and estado["url"] is None:
            estado["url"] = m.group(1)

threading.Thread(target=lanzar_localtunnel, daemon=True).start()
time.sleep(15)

if estado["url"]:
    print(f"\n{'='*60}")
    print(f"URL PÚBLICA PARA EL EQUIPO:")
    print(f"  {estado['url']}")
    print(f"{'='*60}\n")
    print("Compartir al chat de la clase. Cada estudiante:")
    print("  1. Abre el link")
    print("  2. Crea cuenta (email + contraseña)")
    print("  3. Espera asignación de tareas\n")
else:
    print("[WARN] localtunnel no respondió. Revisar logs arriba.")

## 4. Como Manager: crear el proyecto

Abre la URL en una pestaña aparte y haz login (la primera cuenta es admin automáticamente).

**Paso 4.1**: Crear el proyecto
1. **Create Project** → nombre: `Placas Arca`
2. *Labeling Setup* → **Object Detection with Bounding Boxes**
3. Agregar 1 label: `placa` (color rojo)

**Paso 4.2**: Subir las imágenes
1. **Import Data** → arrastra el ZIP de 100 imágenes (mismo de clase 25)
2. URL pública del ZIP: https://raw.githubusercontent.com/cmosquerat/arca-diplomado/main/clase-25/plates_unlabeled.zip

**Paso 4.3**: Invitar al equipo
1. **Members** → invitar a estudiantes con sus emails
2. Asignarles roles (Annotator/Reviewer)

**Paso 4.4**: Asignar lotes
1. Data Manager → seleccionar 5-10 imágenes → **Assign**

## 5. Exportar dataset etiquetado

Cuando todos terminen:

1. **Export** → formato **YOLO**
2. Descargar ZIP
3. Sube el ZIP a la carpeta compartida de Drive
4. Cambiar al Notebook B (`Clase_26_Detection.ipynb`) para fine-tunear

In [ ]:
# Bonus: descargar el dataset programáticamente vía Label Studio SDK
# (solo si quieres automatizar; el botón Export también funciona)

# from label_studio_sdk import Client
# ls = Client(url="http://localhost:8080", api_key="TU_API_KEY")
# project = ls.get_project(PROJECT_ID)
# project.export_tasks(export_type="YOLO", export_location="/content/yolo_export.zip")
print("Alternativa programática (opcional)")